# LABORATORIO 7: VERIFICAR SU CÓDIGO

## Prepararse para verificaciones de código

Para comenzar, reunamos la **información de contexto** que usaremos durante este laboratorio. 

- Haga clic en el botón **Start** para activar este cuaderno.

- Ejecute la siguiente celda Python.


#### :warning: Debe volver a ejecutar la celda a continuación siempre que se inicie una nueva sesión para este cuaderno para configurar “variables” para usar en celdas posteriores. :warning:


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'
print('Su información de contexto actual:')
print('---------------------------------')
print(session)
print('Su usuario actual es' + user)

### Usar INFORMATION_SCHEMA para consultar metadatos 🥋

La palabra “metadatos” significa “datos sobre datos”. 

El esquema `INFORMATION_SCHEMA` creado en cada base de datos de Snowflake contiene metadatos. En otras palabras, contiene estadísticas sobre la cantidad de bases de datos, esquemas, tablas, vistas y más. También contiene datos sobre los nombres de objetos y otros detalles del objeto. 

Podemos usar `INFORMATION_SCHEMA` para comprobar una vez más nuestro trabajo y asegurarnos de haber completado las tareas del modo correcto. 

Obtengamos una lectura de todos los esquemas presentes en la base de datos **(animal)_GARDEN_PLANTS** con la que ha estado trabajando consultando su `INFORMATION_SCHEMA`.


In [ ]:
SELECT * 
FROM {{user}}_garden_plants.information_schema.schemata;

### Usar código para comprobar su trabajo 🥋 

Se le pidió que configure tres esquemas en su base de datos **(animal)_GARDEN_PLANTS**. También se le pidió que eliminara un esquema. Ejecutemos un código para comprobar si se han completado estas tareas.

💡 **CONSEJO**: El código en la siguiente verificación utiliza estructuras [CTE (expresión de tabla común)](https://docs.snowflake.com/en/user-guide/queries-cte). Puede pensar que una CTE es una “vista temporal” que puede usarse en una instrucción. Las CTE son especialmente útiles para desglosar instrucciones SQL complejas, lo que permite leerlas y administrarlas con más facilidad. Al organizar la lógica en distintas partes reutilizables, las CTE simplifican la estructura de la consulta, mejoran la claridad y facilitan su mantenimiento.


In [ ]:
WITH schema_check_1 AS (
    -- ¿Existen los siguientes tres esquemas?
    SELECT COUNT(*) AS count_1
    FROM {{user}}_garden_plants.information_schema.schemata
    WHERE schema_name IN ('FLOWERS','VEGGIES','FRUITS')
),
schema_check_2 AS (   
    -- el siguiente esquema NO DEBE existir (recuento cero)
    SELECT COUNT(*) AS count_2
    FROM {{user}}_garden_plants.information_schema.schemata
    WHERE schema_name = ('PUBLIC')
) 
SELECT IFF((count_1=3) AND (count_2=0),'\u2705 Correcto','\u26D4 Incorrecto. Por favor revise e intente de nuevo') AS schema_check
from schema_check_1, schema_check_2;

## ¿Cuántos esquemas tiene su base de datos `(animal)_GARDEN_PLANTS`? 

### ¿Qué hice mal? 📓 

¿Ejecutó la consulta arriba y el único resultado fue un informe de falla? Aquí describimos algunos errores posibles:

- Hay un error de tipografía en el nombre del esquema, como “WEGGIES” en lugar de “VEGGIES”.

- Incluyó los esquemas en la base de datos equivocada, como UTIL_DB, en lugar de (user)_GARDEN_PLANTS.

- No configuró su rol de modo que pueda ver los objetos, dado que usó `(animal)_LEARNER_RL` para crearlos, pero su hoja de trabajo está configurada en `PUBLIC`. 

---

### ¿Cómo puedo corregir esto? 📓 

**Error gramatical**: `ALTER SCHEMA (animal)_GARDEN_PLANTS.WEGGIES RENAME TO (animal)_GARDEN_PLANTS.VEGGIES;`

**Lugar equivocado**: `ALTER SCHEMA DEMO_DB.VEGGIES RENAME TO (animal)_GARDEN_PLANTS.VEGGIES;`

**No se puede encontrar**: Cambie la configuración del rol en su hoja de trabajo o transfiera la propiedad de su objeto. 


### Verificar los esquemas por nombre 🥋 

Ahora verifiquemos para saber si los esquemas que creó tienen los nombres correctos. Es posible que haya asignado nombres diferentes a sus esquemas si este código no devuelve **tres** filas. 


In [ ]:
SELECT schema_name 
FROM {{user}}_GARDEN_PLANTS.INFORMATION_SCHEMA.SCHEMATA
WHERE schema_name IN ('FLOWERS','FRUITS','VEGGIES');

## Compruebe su trabajo 🔎

### :mag_right: Comprobación 1 (OB01) 🔎

- ¿Creó tres esquemas de la base de datos Garden Plant (**FLOWERS**, **VEGGIES**, **FRUITS**)?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB01', '{{user}}');

### :mag_right: Comprobación 2 (OB02) 🔎

- ¿Eliminó (drop) el esquema de la base de datos Garden Plant con el nombre **PUBLIC**?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB02', '{{user}}');

### :mag_right: Comprobación 3 (OB03) 🔎

- ¿Creó la tabla **ROOT_DEPTH** en el esquema **VEGGIES** de su base de datos Garden Plant?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB03', '{{user}}');

### Use el historial de consultas para revisar los resultados de su prueba 📓

Snowflake conserva un registro de consultas e instrucciones ejecutados en el sistema, conocido como Historial de consultas (Query History), y proporciona la interfaz de usuario y formas programáticas de acceder a este. El Query History es un lugar conveniente en el que ver las consultas que usted (u otros, si tiene los privilegios) ejecutó con el tiempo.

Podemos acceder al Query History desde una celda en un cuaderno de Snowflake.


### Acceda al Query History desde una celda SQL del cuaderno de Snowflake 🥋

1. Desplácese por la lectura de tiempo de ejecución de la consulta en la celda SQL **Check 3** que acaba de ejecutar.
1. Verá que aparece el mensaje **View run details**.
1. Haga clic en la lectura de tiempo de ejecución de la consulta y aparecerá un cuadro de diálogo.
1. Haga clic en el campo UUID del **ID** azul, que incluye un enlace a la página Query History de Snowsight.
1. Esto abrirá la página Historial de consultas de Snowsight en una nueva ventana del navegador.

![Acceder a Query History (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_query_history_1_v2.png)

1. Esto se abrirá en la pantalla **Query Profile**, en la que puede revisar los pasos de ejecución para su consulta.
1. Haga clic en la pestaña **Query Details** en la parte superior de la pantalla.
1. Revise los distintos detalles relacionados con la ejecución de la consulta.
1. Compruebe la sección **Results** al final de la página. 

![Detalles de consulta (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_query_details_1_v2.png)

:warning: **NO CONTINUAR** después de este punto, a menos que vea una marca de verificación verde ✅ en la sección Results de los detalles de la consulta (Query History) para el Check 3\. :warning:


### Acceder Query History mediante programación 🥋

También podemos recuperar información de Query History mediante código, con opciones disponibles para Python y SQL. 

`INFORMATION_SCHEMA` contiene una colección de [funciones de tabla](https://docs.snowflake.com/en/sql-reference/functions/query_history) que se pueden utilizar para consultar el Query History de Snowflake en varias dimensiones. En el siguiente ejemplo, usaremos `QUERY_HISTORY_BY_USER()` para devolver consultas emitidas por un usuario especificado (usted) en los últimos siete días. 

Vamos a identificar la operación `DELETE` que ejecutó en **Laboratorio 6: El Asistente de carga y Snowflake Marketplace** que eliminó una única fila de la tabla **VEGETABLE_DETAILS** (...plant_name = 'Spinach' AND root_depth_code = 'D').


In [ ]:
SELECT *
FROM TABLE(information_schema.query_history_by_user(
    USER_NAME => '{{user}}',
    RESULT_LIMIT => 10000
))
WHERE query_type = 'DELETE'
AND execution_status = 'SUCCESS'
ORDER BY end_time DESC
LIMIT 1;

Guardemos el ID de consulta para esta operación en una variable de SQL que utilizaremos más tarde en este laboratorio.

**:warning: `INFORMATION_SCHEMA.QUERY_HISTORY_BY_USER` y sus variaciones solo retienen datos por siete días. Si ejecuta la consulta anterior fuera de esta ventana, entonces no se devolverán resultados. Deberá completar nuevamente los pasos en el Laboratorio 6\. :warning:**


In [ ]:
SET delete_query_id = (
    SELECT query_id
    FROM TABLE(information_schema.query_history_by_user(
        USER_NAME => '{{user}}',
        RESULT_LIMIT => 10000
    ))
    WHERE query_type = 'DELETE'
    AND execution_status = 'SUCCESS'
    ORDER BY end_time DESC
    LIMIT 1
);

SELECT $delete_query_id;

### :mag_right: Comprobación 4 (OB04) 🔎

- ¿Su base de datos **(animal)_UTIL_DB** incluye dos esquemas (y solo dos)?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB04', '{{user}}');

### :mag_right: Comprobación 5 (OB05) 🔎

- ¿Creó la tabla **VEGETABLE_DETAILS** en el esquema **VEGGIES** de la base de datos Garden Plant?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB05', '{{user}}');

### :mag_right: Comprobación 6 (OB06) 🔎

- ¿La tabla **ROOT_DEPTH** tiene **tres** filas?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB06', '{{user}}');  

### :mag_right: Comprobación 7 (OB07) 🔎

- ¿La tabla **VEGETABLE_DETAILS** tiene **41** filas?
- Abra el procedimiento almacenado de calificación para comprobar su trabajo.


In [ ]:
CALL common_db.resources.local_grader('OB07', '{{user}}'); 

## Time Travel 📓

Time Travel de Snowflake permite acceder a datos históricos (es decir, datos que cambiaron o se eliminaron) en cualquier punto dentro de un período definido. Para admitir Time Travel, hay varias [extensiones SQL](https://docs.snowflake.com/en/user-guide/data-time-travel#time-travel-sql-extensions) disponibles.

Cuando se modifican datos en una tabla, lo que incluye eliminar datos o eliminar(drop) un objeto que contiene datos, Snowflake preserva el estado de los datos antes de la actualización. Un parámetro llamado `DATA_RETENTION_TIME_IN_DAYS` especifica la cantidad de días durante los cuales se preservan estos datos históricos y, por ende, el plazo en el cual se pueden realizar operaciones Time Travel (`SELECT`, `CREATE` … `CLONE`, `UNDROP`) en los datos.


### Ver una versión histórica de la tabla **VEGETABLE_DETAILS** 🥋

Cuando se creó la tabla **VEGETABLE_DETAILS** al principio del **Laboratorio 6: El Asistente de carga y Snowflake Marketplace**, esta se configuró con una conservación de datos de siete días. Esto significa que cualquier cambio realizado a la tabla se conserva por un plazo de siete días. Esto nos permite volver en el tiempo y navegar para ver los datos asociados a la tabla en un punto en particular, ya sea de forma cronológica o antes de que se ejecutara una operación específica en la tabla.

En el **Laboratorio 6**, eliminó una segunda fila Espinaca(Spinach) de la tabla **VEGETABLE_DETAILS**. Anteriormente en este laboratorio, identificamos el ID de consulta asociada a esa operación y la guardamos en una variable local: `$delete_query_id`. Podemos usar Time Travel para ver los datos antes de esta operación. 

Ejecute el siguiente código para ver y etiquetar las filas Spinach antes de que `DELETE` **las uniera** con los datos Spinach desde la versión actual de la tabla. Tenga en cuenta la [sintaxis especial](https://docs.snowflake.com/en/user-guide/data-time-travel#querying-historical-data) utilizada para consultar datos históricos mediante Time Travel: `BEFORE( STATEMENT => $delete_query_id )` 


In [ ]:
-- Consulta de viaje en el tiempo
SELECT plant_name, root_depth_code, 'HISTORICAL (Time Travel)' as table_version 
FROM {{user}}_garden_plants.veggies.vegetable_details
BEFORE( STATEMENT => $delete_query_id )
WHERE plant_name = 'Spinach'

UNION

-- consulta de versión actual
SELECT plant_name, root_depth_code, 'CURRENT' as table_version
FROM {{user}}_garden_plants.veggies.vegetable_details
WHERE plant_name = 'Spinach';

## Ponga a prueba sus conocimientos :mag_right:

Compruebe sus conocimientos con las preguntas interactivas que aparecen a continuación. Cada celda `RUN_THIS_QUIZ_QUESTION_` tiene un widget Streamlit que presenta una pregunta de opción múltiple sobre la funcionalidad de Snowflake.  

**Instrucciones:**  
1. Pase el cursor sobre la celda del cuaderno para revelar controles adicionales.
1. Haga clic en el botón ▶️ **Reproducir** en el lado derecho de cada celda del cuestionario para ejecutarla.  
1. Seleccione su respuesta entre las opciones que se proporcionan.  
1. Revise los comentarios antes de continuar. 

💡 **Nota:** Puede expandir la celda para ver el código si tiene curiosidad, pero no es obligatorio. Estos cuestionarios no son obligatorios: están disponibles para desafiar su transferencia de conocimientos y proporcionarle la oportunidad de practicar.  


In [ ]:
import streamlit as st
st.divider()
question = "¿Cuál es la definición de Metadatos?"
options = ["Seleccione una opción a continuación...",
           "A) Datos sobre meta", 
           "B) Datos que están por encima de otros datos", 
           "C) Datos sobre datos"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...":
        ''
    else:
        answer = 'd29cf950f90eec2b6adc5dc386350060'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "¿Dónde almacena Snowflake algunos de sus metadatos?"
options = ["Seleccione una opción a continuación...",
           "A) La base de datos INFORMATION_DB de cada cuenta", 
           "B) El esquema METADATA_SCHEMA de cada base de datos", 
           "C) El esquema INFO_METADATA de cada base de datos",
           "D) El esquema INFORMATION_SCHEMA de cada base de datos"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...":
        ''
    else:
        answer = 'd16dd4fe43e16e66f5f226ca875f63db'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "Si queremos comprobar los esquemas que creamos, ¿Por qué no simplemente mirar con nuestros propios ojos para ver si están allí?"
options = ["Seleccione una opción a continuación...",
           "A) Porque si escribimos algo mal, es posible que no notemos el problema de nombre, pero la comprobación mediante código podría detectarlo", 
           "B) Porque podemos querer automatizar la comprobación por alguna razón", 
           "C) Porque las comprobaciones manuales son más divertidas que la codificación"]

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...":
        ''
    else:
        answer = '0e66f786d2541f2250d041ee34443521'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## Siguientes pasos

Si pasó las comprobaciones de calificaciones (de 1 a 7) y respondió correctamente las preguntas de la **Prueba de conocimientos**, continúe al siguiente cuaderno cuando se lo indique el instructor de Snowflake.
